# 08 — Explainability Analysis
SHAP + attention maps + gate weight analysis.

In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import json, warnings
import numpy as np
import pandas as pd
import torch
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from datasets import load_dataset
from transformers import DistilBertTokenizerFast

from src.fusion_model import HybridSalesPredictor
from src.explainability import HybridExplainer

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
MODELS_DIR = ROOT / 'models'
FIGURES_DIR = ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)
DEVICE = 'cpu'
SAMPLE_SIZE = 500
MAX_LEN = 128
USE_XGBOOST = False  # Disabled by default: xgboost fit crashes this kernel.
LOAD_HYBRID_MODEL = False  # Set True only when running attention/gate sections.

print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])
print('NumPy:', np.__version__, '| Torch:', torch.__version__)
print('Setup done.')

Python executable: /Users/spylook/predictive-sales-analytics-engine/.venv/bin/python
Python version: 3.14.4
NumPy: 2.4.4 | Torch: 2.11.0
Setup done.


## Load data & models

In [2]:
print('[step 1] Streaming dataset rows from HF...')
rows = []
try:
    ds_stream = load_dataset(
        'DeepMostInnovations/saas-sales-conversations',
        split='train',
        streaming=True,
    )
    for i, row in enumerate(ds_stream):
        rows.append(row)
        if i + 1 >= SAMPLE_SIZE:
            break
except Exception as e:
    raise RuntimeError(f'Failed to stream dataset from HF Hub: {e}')

if not rows:
    raise RuntimeError('No rows loaded from dataset; cannot continue.')

print(f'[step 2] Building dataframe from {len(rows)} rows...')
df = pd.DataFrame(rows).reset_index(drop=True)
y = df['outcome'].astype(int).values

print('[step 3] Building tabular features...')
num_cols = [c for c in ['conversation_length', 'customer_engagement', 'sales_effectiveness'] if c in df.columns]
cat_cols = [c for c in ['product_type', 'conversation_style', 'communication_channel'] if c in df.columns]
tab_df = df[num_cols + cat_cols].copy()
if 'customer_engagement' in tab_df.columns and 'sales_effectiveness' in tab_df.columns:
    tab_df['eng_eff_ratio'] = tab_df['customer_engagement'] / (tab_df['sales_effectiveness'] + 0.001)
fin_num = [c for c in tab_df.columns if c not in cat_cols]
scaler = StandardScaler()
X_num = scaler.fit_transform(tab_df[fin_num].fillna(0).values)
if cat_cols:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_cat = ohe.fit_transform(tab_df[cat_cols].fillna('unknown').astype(str))
    X_tab = np.hstack([X_num, X_cat])
    feature_names = fin_num + ohe.get_feature_names_out(cat_cols).tolist()
else:
    X_tab = X_num
    feature_names = fin_num

print('[step 4] Training safe tree model...')
if USE_XGBOOST:
    # Optional: keep this branch only if your environment can train xgboost safely.
    import xgboost as xgb
    print('Training XGBoost (USE_XGBOOST=True)...')
    model = xgb.XGBClassifier(
        n_estimators=80,
        max_depth=6,
        learning_rate=0.1,
        eval_metric='logloss',
        random_state=SEED,
        verbosity=0,
    )
else:
    print('Training RandomForest (default safe path)...')
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        random_state=SEED,
        n_jobs=-1,
    )

model.fit(X_tab, y)
# Keep variable name for downstream notebook cells.
xgb_model = model

print('[step 5] Optional hybrid model load...')
model_path = MODELS_DIR / 'hybrid_model.pt'
hybrid_model = None
if LOAD_HYBRID_MODEL and model_path.exists():
    try:
        hybrid_model = torch.load(str(model_path), map_location=DEVICE)
        hybrid_model.eval()
        print('Loaded hybrid model.')
    except Exception as e:
        print(f'Could not load hybrid model: {e}')
        print('Continuing with XGBoost-only explainability.')
elif not LOAD_HYBRID_MODEL:
    print('Skipping hybrid model load (LOAD_HYBRID_MODEL=False).')
else:
    print('hybrid_model.pt not found — run notebook 06 first. Skipping DL explainability.')

print(f'Loaded rows: {len(df)}')
print(f'X_tab: {X_tab.shape}  Features: {len(feature_names)}')

[step 1] Streaming dataset rows from HF...


[step 2] Building dataframe from 500 rows...
[step 3] Building tabular features...
[step 4] Training safe tree model...
Training RandomForest (default safe path)...
[step 5] Optional hybrid model load...
Skipping hybrid model load (LOAD_HYBRID_MODEL=False).
Loaded rows: 500
X_tab: (500, 22)  Features: 22


## SHAP — Feature importance

In [3]:
print('[1/4] SHAP summary plot...')
explainer = shap.TreeExplainer(xgb_model)
shap_vals = explainer.shap_values(X_tab[:500])  # sample 500 for speed

fig, ax = plt.subplots(figsize=(9, 6))
shap.summary_plot(shap_vals, X_tab[:500], feature_names=feature_names,
                   show=False, max_display=15)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()
print('  Saved shap_summary.png')

[1/4] SHAP summary plot...
  Saved shap_summary.png


## SHAP waterfall for individual examples

In [6]:
print('[2/4] SHAP waterfall for 3 individual samples...')
sample_indices = [0, 1, 2]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Normalize SHAP output to (n_samples, n_features) for plotting.
# Different tree models return different shapes:
# - list[class] of (n_samples, n_features)
# - (n_samples, n_features)
# - (n_samples, n_features, n_classes)
if isinstance(shap_vals, list):
    shap_plot_vals = shap_vals[1] if len(shap_vals) > 1 else shap_vals[0]
else:
    shap_plot_vals = np.asarray(shap_vals)
    if shap_plot_vals.ndim == 3:
        # Prefer positive class if available.
        class_idx = 1 if shap_plot_vals.shape[2] > 1 else 0
        shap_plot_vals = shap_plot_vals[:, :, class_idx]

for ax_i, si in enumerate(sample_indices):
    sv = np.asarray(shap_plot_vals[si]).reshape(-1)
    sorted_pairs = sorted(zip(feature_names, sv), key=lambda x: abs(float(x[1])), reverse=True)[:8]
    names_s = [p[0] for p in sorted_pairs]
    values_s = [float(p[1]) for p in sorted_pairs]
    colors = ['#e74c3c' if v > 0 else '#3498db' for v in values_s]
    axes[ax_i].barh(names_s, values_s, color=colors)
    axes[ax_i].axvline(0, color='black', linewidth=0.8)
    axes[ax_i].set_title(f'Sample {si} (y={y[si]}, pred={xgb_model.predict([X_tab[si]])[0]})')
    axes[ax_i].set_xlabel('SHAP Value')

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'shap_waterfall_samples.png'), dpi=150)
plt.show()
print('  Saved shap_waterfall_samples.png')

[2/4] SHAP waterfall for 3 individual samples...
  Saved shap_waterfall_samples.png


## Attention map (requires hybrid model)

In [7]:
if hybrid_model is None:
    print('[3/4] Skipping attention map — hybrid model not loaded.')
else:
    print('[3/4] Attention heatmap for example conversations...')
    tok = DistilBertTokenizerFast.from_pretrained(
        str(MODELS_DIR / 'tokenizer') if (MODELS_DIR / 'tokenizer').exists() else 'distilbert-base-uncased'
    )
    text_col = 'full_text' if 'full_text' in df.columns else 'conversation'
    example_texts = df[text_col].fillna('').astype(str).head(3).tolist()

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax_i, text in enumerate(example_texts):
        enc = tok([text], max_length=MAX_LEN, padding='max_length',
                   truncation=True, return_tensors='pt')
        # Dummy leaf for attention extraction
        cfg = json.load(open(MODELS_DIR / 'feature_config.json')) if (MODELS_DIR / 'feature_config.json').exists() else {'leaf_dim': 100}
        leaf_dim = cfg.get('leaf_dim', 100)
        dummy_leaf = torch.zeros(1, leaf_dim)
        if hybrid_model.tab_projection is not None and list(hybrid_model.tab_projection.parameters())[0].shape[1] == leaf_dim:
            with torch.no_grad():
                _, _, attn_w = hybrid_model(dummy_leaf, enc['input_ids'], enc['attention_mask'])
            w = attn_w[0].cpu().numpy()
            axes[ax_i].bar(range(len(w)), w, color='coral', alpha=0.7)
            axes[ax_i].set_title(f'Conv {ax_i} (y={y[ax_i]})')
            axes[ax_i].set_xlabel('Token position')
            axes[ax_i].set_ylabel('Attention weight')
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / 'attention_heatmaps.png'), dpi=150)
    plt.show()
    print('  Saved attention_heatmaps.png')

[3/4] Skipping attention map — hybrid model not loaded.


## Gate weight analysis

In [8]:
if hybrid_model is None:
    print('[4/4] Skipping gate analysis — hybrid model not loaded.')
else:
    print('[4/4] Gate weight distribution...')
    from torch.utils.data import DataLoader, TensorDataset

    tok = DistilBertTokenizerFast.from_pretrained(
        str(MODELS_DIR / 'tokenizer') if (MODELS_DIR / 'tokenizer').exists() else 'distilbert-base-uncased'
    )
    text_col = 'full_text' if 'full_text' in df.columns else 'conversation'
    texts = df[text_col].fillna('').astype(str).head(200).tolist()
    enc = tok(texts, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')

    cfg = json.load(open(MODELS_DIR / 'feature_config.json')) if (MODELS_DIR / 'feature_config.json').exists() else {'leaf_dim': 100}
    leaf_dim = cfg.get('leaf_dim', 100)
    dummy_leaves = torch.zeros(len(texts), leaf_dim)

    all_gate_means = []
    ds_g = TensorDataset(dummy_leaves, enc['input_ids'], enc['attention_mask'])
    dl_g = DataLoader(ds_g, batch_size=16)
    with torch.no_grad():
        for lb, ids, masks in dl_g:
            try:
                _, gate, _ = hybrid_model(lb, ids, masks)
                all_gate_means.extend(gate.mean(dim=1).numpy().tolist())
            except Exception:
                pass

    if all_gate_means:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(all_gate_means, bins=30, color='steelblue', edgecolor='white')
        ax.axvline(np.mean(all_gate_means), color='red', linestyle='--',
                   label=f'Mean={np.mean(all_gate_means):.3f}')
        ax.set_xlabel('Mean Gate Value (0=trust text, 1=trust tabular)')
        ax.set_ylabel('Count')
        ax.set_title('Gate Weight Distribution — Tabular vs Text Reliance')
        ax.legend()
        plt.tight_layout()
        plt.savefig(str(FIGURES_DIR / 'gate_distribution.png'), dpi=150)
        plt.show()
        print(f'  Mean gate (tabular reliance): {np.mean(all_gate_means):.3f}')
        print('  Saved gate_distribution.png')
    else:
        print('  Could not extract gate values (leaf_dim mismatch — run notebook 06 first).')

print('\nDone ✅')

[4/4] Skipping gate analysis — hybrid model not loaded.

Done ✅
